# PLDStega Google Colab Notebook

Run this notebook in Google Colab with `Runtime > Change runtime type > GPU`. It clones the GitHub repo, installs a pinned Diffusers/Transformers stack while keeping Colab's CUDA PyTorch, runs CPU-safe tests, then runs a small verified PLDStega hide/extract smoke test.

This is an experimental research prototype. Trust a generated stego image only when hide prints `verified=True`.

## 1. Setup Repo

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

REPO_URL = 'https://github.com/kush1999-h/PLDStega_SecurityAssignment.git'
PROJECT_DIR = Path('/content/PLDStega_SecurityAssignment')

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print('Project directory:', PROJECT_DIR)
print('Latest commit:')
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

## 2. Install Dependencies

Do not reinstall PyTorch in Colab. Colab already provides a CUDA PyTorch build matching its GPU image.

In [ ]:
install_cmd = [
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'diffusers==0.30.3',
    'transformers==4.44.2',
    'tokenizers==0.19.1',
    'accelerate==0.33.0',
    'safetensors==0.4.5',
    'huggingface-hub==0.25.2',
    'numpy>=1.24,<2.1',
    'pillow>=10,<12',
    'cryptography>=42,<50',
    'reedsolo==1.7.0',
]
subprocess.run(install_cmd, check=True)

# Colab often preinstalls PEFT. This project does not use PEFT/LoRA, and
# incompatible PEFT + Hugging Face Hub versions can break Diffusers imports.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'], check=False)

## 3. Check GPU And Versions

In [ ]:
check_code = r'''
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TRANSFORMERS_NO_TF'] = '1'
import torch
import importlib.util
import diffusers
import huggingface_hub
import transformers
print('Python:', __import__('sys').version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Colab: Runtime > Change runtime type > GPU')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Diffusers:', diffusers.__version__)
print('Hugging Face Hub:', huggingface_hub.__version__)
print('Transformers:', transformers.__version__)
print('PEFT installed:', importlib.util.find_spec('peft') is not None)
'''
subprocess.run([sys.executable, '-c', check_code], check=True)

## 4. Optional Hugging Face Login

If downloads are rate-limited, create a Colab secret named `HF_TOKEN`.

In [ ]:
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    from huggingface_hub import login
    login(token=token)
    print('Logged in to Hugging Face.')
else:
    print('No HF_TOKEN found. Continuing unauthenticated.')

## 5. Run CPU-Safe Tests

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)

## 6. Configure Smoke Test

These settings are smaller than the full 768px experiment because the current PLDStega path verifies the payload after image generation. Increase parameters only after `verified=True` works.

In [ ]:
MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
PROMPT = 'realistic photo of a man catching fish, natural outdoor light, sharp focus'
NEGATIVE_PROMPT = 'blurry, distorted, deformed hands, extra fingers, low quality'
MESSAGE = 'KOUSHIK'
KEY = 'shared-key'
SEED = 1234

HEIGHT = 512
WIDTH = 512
STEPS = 20
GUIDANCE_SCALE = 7.5

CAPACITY_BYTES = 64
GROUP_SIZE = 9
REPEAT = 1
ECC_SYMBOLS = 16
EMBED_METHOD = 'sign'
EMBED_STRENGTH = 0.10
STABILIZE_ROUNDS = 6
DTYPE = 'float32'
USE_CPU_OFFLOAD = True

OUTPUT = 'outputs/colab_pldstega_smoke.png'
Path('outputs').mkdir(exist_ok=True)
print('Output:', OUTPUT)

## 7. Hide Message

This cell should finish with `verified=True`. If it fails verification, the image should not be trusted as containing a recoverable payload.

In [ ]:
env = os.environ.copy()
env['USE_TF'] = '0'
env['USE_FLAX'] = '0'
env['USE_TORCH'] = '1'
env['TRANSFORMERS_NO_TF'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

hide_cmd = [
    sys.executable, '-m', 'ldstega.cli', 'hide',
    '--mode', 'pldstega',
    '--model', MODEL,
    '--prompt', PROMPT,
    '--negative-prompt', NEGATIVE_PROMPT,
    '--message', MESSAGE,
    '--key', KEY,
    '--seed', str(SEED),
    '--output', OUTPUT,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--steps', str(STEPS),
    '--guidance-scale', str(GUIDANCE_SCALE),
    '--capacity-bytes', str(CAPACITY_BYTES),
    '--group-size', str(GROUP_SIZE),
    '--repeat', str(REPEAT),
    '--ecc-symbols', str(ECC_SYMBOLS),
    '--embed-method', EMBED_METHOD,
    '--embed-strength', str(EMBED_STRENGTH),
    '--dtype', DTYPE,
    '--stabilize-rounds', str(STABILIZE_ROUNDS),
]
if USE_CPU_OFFLOAD:
    hide_cmd.append('--cpu-offload')

print(' '.join(hide_cmd))
subprocess.run(hide_cmd, check=True, env=env)

In [ ]:
from IPython.display import Image as DisplayImage, display
display(DisplayImage(filename=OUTPUT))

## 8. Promptless Extract

Run this only after hide prints `verified=True`.

In [ ]:
extract_cmd = [
    sys.executable, '-m', 'ldstega.cli', 'extract',
    '--mode', 'pldstega',
    '--image', OUTPUT,
    '--key', KEY,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--capacity-bytes', str(CAPACITY_BYTES),
    '--group-size', str(GROUP_SIZE),
    '--repeat', str(REPEAT),
    '--ecc-symbols', str(ECC_SYMBOLS),
    '--embed-method', EMBED_METHOD,
    '--dtype', DTYPE,
]
result = subprocess.run(extract_cmd, check=True, capture_output=True, text=True, env=env)
print('Recovered message:')
print(result.stdout.strip())

## 9. Troubleshooting

- CUDA OOM: set `HEIGHT = WIDTH = 512`, keep `USE_CPU_OFFLOAD = True`, reduce `STABILIZE_ROUNDS`.
- Verification fails: increase `EMBED_STRENGTH`, reduce `CAPACITY_BYTES`, or reduce `STEPS`.
- Do not extract from images generated with `--allow-unverified` unless you are debugging.